In [ ]:
import random
import pandas as pd
import numpy as np
import os
import IPython.display as ipd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings(action='ignore') 

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
# 1. 건물 정보 데이터 로드
building_info = pd.read_csv('building_info.csv')
# 2. '건물번호'를 기준으로 train과 test 데이터에 건물 정보 병합
train = pd.merge(train, building_info, on='건물번호', how='left')
test = pd.merge(test, building_info, on='건물번호', how='left')

In [ ]:
categorical_cols = train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

In [ ]:
# 1. 처리할 컬럼 리스트
zero_cols = ['강수량(mm)', '일조(hr)', '일사(MJ/m2)']

# 2. Train 데이터 처리
for col in zero_cols:
    train[col] = train[col].fillna(0)

# 3. Test 데이터 처리 (없으면 생성, 있으면 결측치 채우기)
for col in zero_cols:
    if col not in test.columns:
        # 컬럼이 아예 없으면 0으로 가득 채운 새 컬럼 생성
        test[col] = 0
    else:
        # 컬럼이 있다면 결측치만 0으로 채움
        test[col] = test[col].fillna(0)

# 4. 선형 보간 (기온, 풍속, 습도)
cols_to_interp = ['기온(C)', '풍속(m/s)', '습도(%)']
train[cols_to_interp] = train[cols_to_interp].interpolate(method='linear')
test[cols_to_interp] = test[cols_to_interp].interpolate(method='linear')

# 결과 확인
print(train.isnull().sum())
print(test.isnull().sum())

num_date_time    0
건물번호             0
일시               0
기온(C)            0
강수량(mm)          0
풍속(m/s)          0
습도(%)            0
일조(hr)           0
일사(MJ/m2)        0
전력소비량(kWh)       0
건물유형             0
연면적(m2)          0
냉방면적(m2)         0
태양광용량(kW)        0
ESS저장용량(kWh)     0
PCS용량(kW)        0
dtype: int64
num_date_time    0
건물번호             0
일시               0
기온(C)            0
강수량(mm)          0
풍속(m/s)          0
습도(%)            0
건물유형             0
연면적(m2)          0
냉방면적(m2)         0
태양광용량(kW)        0
ESS저장용량(kWh)     0
PCS용량(kW)        0
일조(hr)           0
일사(MJ/m2)        0
dtype: int64


In [ ]:
for data in [train, test]:
    data['일시'] = pd.to_datetime(data['일시'])
    data['year'] = data['일시'].dt.year
    data['month'] = data['일시'].dt.month
    data['day'] = data['일시'].dt.day
    data['hour'] = data['일시'].dt.hour
    data['dayofweek'] = data['일시'].dt.dayofweek

In [ ]:
# =========================================================================
# [추가] team_ex10: 임시 휴무일 통계적 Drop 로직
# =========================================================================
print(f"🧹 임시 휴무 Drop 전 Train 크기: {train.shape}")

# 1. 계산을 위해 date 컬럼 임시 생성
train['date'] = train['일시'].dt.date

# 2. 건물별 + 일별 '총 전력소비량' 계산
daily_power = train.groupby(['건물번호', 'date'])['전력소비량(kWh)'].sum().reset_index()
daily_power.rename(columns={'전력소비량(kWh)': '일별총전력'}, inplace=True)

# 3. 건물별 하루 전력소비량의 '중앙값(평소 전력량)' 계산 (평균보다 이상치에 강함)
building_median = daily_power.groupby('건물번호')['일별총전력'].median().reset_index()
building_median.rename(columns={'일별총전력': '건물평소전력'}, inplace=True)

# 4. 데이터 병합 및 비율 계산
daily_power = pd.merge(daily_power, building_median, on='건물번호')
daily_power['전력비율'] = daily_power['일별총전력'] / daily_power['건물평소전력']

# 5. Drop 기준: 평소 전력량의 30% (0.3) 이하만 쓴 날을 '휴무'로 간주
# (이 0.3 이라는 수치는 실험을 통해 0.2 나 0.4 로 조절 가능합니다)
closed_days = daily_power[daily_power['전력비율'] < 0.3][['건물번호', 'date']]
closed_days['is_closed'] = 1

# 6. 원본 train 데이터에 휴무일 정보 매핑
train = pd.merge(train, closed_days, on=['건물번호', 'date'], how='left')
train['is_closed'] = train['is_closed'].fillna(0)

# 7. 휴무일(1)인 행을 Drop 하고 정상(0)인 행만 남김
train = train[train['is_closed'] == 0].reset_index(drop=True)

# 8. 임시로 만든 컬럼들 청소
train.drop(columns=['is_closed', 'date'], inplace=True)

print(f"✨ 임시 휴무 Drop 후 Train 크기: {train.shape}")
# =========================================================================

🧹 임시 휴무 Drop 전 Train 크기: (204000, 21)
✨ 임시 휴무 Drop 후 Train 크기: (203904, 21)


In [ ]:
print(train['건물유형'].nunique())

12


In [ ]:
train['hour_sin'] = np.sin(2*np.pi*train['hour']/24)
train['hour_cos'] = np.cos(2*np.pi*train['hour']/24)

test['hour_sin'] = np.sin(2*np.pi*test['hour']/24)
test['hour_cos'] = np.cos(2*np.pi*test['hour']/24)

In [ ]:
# 6월 6일 평균 대비 6월 7일 평균 전력의 비율 계산
#june_6 = train[train['일시'].dt.strftime('%m-%d') == '06-06'].groupby('건물번호')['전력소비량(kWh)'].mean()
#june_7 = train[train['일시'].dt.strftime('%m-%d') == '06-07'].groupby('건물번호')['전력소비량(kWh)'].mean()

#drop_ratio = (june_6 / june_7)
# 비율이 0.7 미만(30% 이상 급감)인 건물들만 보기
#sensitive_buildings = drop_ratio[drop_ratio < 0.7].index.tolist()

#print(f"공휴일에 민감한 건물 번호: {sensitive_buildings}")

In [ ]:
train['is_holiday'] = train['일시'].dt.strftime('%m-%d').isin(['06-01', '06-06', '08-15']).astype(int)
train['is_holiday'] = ((train['is_holiday'] == 1) | (train['dayofweek'] >= 5)).astype(int)

In [ ]:
train['is_weekend'] = (train['dayofweek'] >= 5).astype(int)
test['is_weekend'] = (test['dayofweek'] >= 5).astype(int)

In [ ]:
# 불쾌지수 계산 함수 정의
def calculate_di(temp, humid):
    return 1.8 * temp - 0.55 * (1 - humid/100) * (1.8 * temp - 26) + 32

# 피처 생성
train['DI'] = calculate_di(train['기온(C)'], train['습도(%)'])
test['DI'] = calculate_di(test['기온(C)'], test['습도(%)'])

# 확인용
print(train[['기온(C)', '습도(%)', 'DI']].head())

   기온(C)  습도(%)        DI
0   18.6   42.0  63.09388
1   18.0   45.0  62.46400
2   17.7   45.0  62.08735
3   16.7   48.0  60.89884
4   18.4   43.0  62.88788


In [ ]:
for df in [train, test]:
    df['date'] = df['일시'].dt.date 

    # 건물별 + 날짜별로 그룹화하여 평균, 최대 계산
    df['평균기온'] = df.groupby(['건물번호', 'date'])['기온(C)'].transform('mean')
    df['최대기온'] = df.groupby(['건물번호', 'date'])['기온(C)'].transform('max')

    # 사용 후 date 컬럼 제거 (선택 사항)
    df.drop(columns=['date'], inplace=True)

In [ ]:
# 체감온도(Apparent Temperature) 계산 함수 정의
def calculate_at(temp, humid, wind):
    # 수증기압(e) 계산
    e = (humid / 100) * 6.105 * np.exp(17.27 * temp / (237.7 + temp))
    # 체감온도 공식 (여름철 풍속 영향 반영)
    return 1.04 * temp + 0.2 * e - 0.65 * wind - 2.7

# [기존 코드의 피처 생성 부분에 추가]
# Train 데이터
train['AT'] = calculate_at(train['기온(C)'], train['습도(%)'], train['풍속(m/s)'])

# Test 데이터
test['AT'] = calculate_at(test['기온(C)'], test['습도(%)'], test['풍속(m/s)'])

# 확인용
print(train[['기온(C)', '습도(%)', '풍속(m/s)', 'AT']].head())

   기온(C)  습도(%)  풍속(m/s)         AT
0   18.6   42.0      0.9  17.854843
1   18.0   45.0      1.1  17.158145
2   17.7   45.0      1.5  16.551526
3   16.7   48.0      1.4  15.578997
4   18.4   43.0      2.8  16.431747


In [ ]:
# =========================================================================
# [추가] team_ex11: 건물별 전력 사용 패턴 기반 클러스터링 (성향 분석)
# =========================================================================
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("🧬 건물의 유전자(사용 패턴) 분석 중...")

# 1. 건물별 특징(Feature) 추출용 데이터프레임 생성
building_profiles = []

for b_num in train['건물번호'].unique():
    b_data = train[train['건물번호'] == b_num]
    
    # 성향 A: 건물 체급 (평균 사용량)
    mean_p = b_data['전력소비량(kWh)'].mean()
    
    # 성향 B: 냉방 민감도 (기온과 전력량의 상관관계)
    temp_corr = b_data[['전력소비량(kWh)', '기온(C)']].corr().iloc[0, 1]
    
    # 성향 C: 운영 패턴 (주말/주중 전력 사용 비율)
    # 0에 가까우면 주말에 완전히 쉬는 건물, 1에 가까우면 24시간 풀가동 건물
    weekday_p = b_data[b_data['dayofweek'] < 5]['전력소비량(kWh)'].mean()
    weekend_p = b_data[b_data['dayofweek'] >= 5]['전력소비량(kWh)'].mean()
    op_pattern = weekend_p / (weekday_p + 1e-6)
    
    building_profiles.append([b_num, mean_p, temp_corr, op_pattern])

df_profiles = pd.DataFrame(building_profiles, columns=['건물번호', 'mean_p', 'temp_corr', 'op_pattern'])

# 2. 클러스터링 (K-Means) 실행
# 특징들의 단위가 다르므로 스케일링 필수!
scaler = StandardScaler()
features_scaled = scaler.fit_transform(df_profiles[['mean_p', 'temp_corr', 'op_pattern']])

# 4개 그룹으로 분류 (건물 유형이 다양하므로 4~6개가 적당합니다)
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_profiles['building_cluster'] = kmeans.fit_predict(features_scaled)

# 3. 원본 데이터(train, test)에 클러스터 정보 합치기
train = pd.merge(train, df_profiles[['건물번호', 'building_cluster']], on='건물번호', how='left')
test = pd.merge(test, df_profiles[['건물번호', 'building_cluster']], on='건물번호', how='left')

# 카테고리 형으로 변환 (XGBoost가 범주형으로 인식하도록)
train['building_cluster'] = train['building_cluster'].astype('category')
test['building_cluster'] = test['building_cluster'].astype('category')

print(f"✅ 클러스터링 완료! 분류 현황:\n{df_profiles['building_cluster'].value_counts().sort_index()}")
# =========================================================================

🧬 건물의 유전자(사용 패턴) 분석 중...
✅ 클러스터링 완료! 분류 현황:
building_cluster
0    33
1    45
2     4
3    18
Name: count, dtype: int64


In [ ]:
# [1] 공통 피처 리스트 정의 (순서가 중요합니다!)
features = [
    '건물번호','building_cluster', '기온(C)', '강수량(mm)', '풍속(m/s)', '습도(%)', 'DI','평균기온', '최대기온', 'AT',
    'month', 'day', 'hour', 'dayofweek', 'hour_sin', 'hour_cos', 'is_holiday','is_weekend', '건물유형', '연면적(m2)', '냉방면적(m2)', '태양광용량(kW)', 'ESS저장용량(kWh)', 'PCS용량(kW)'
]

# [2] Test 데이터 전처리 (Train은 이미 되어 있다고 가정)
test['일시'] = pd.to_datetime(test['일시'])
test['month'] = test['일시'].dt.month
test['day'] = test['일시'].dt.day
test['hour'] = test['일시'].dt.hour
test['dayofweek'] = test['일시'].dt.dayofweek

# 공휴일 생성 (현충일, 광복절 + 주말) # 06-01 선거일 
test['is_holiday'] = test['일시'].dt.strftime('%m-%d').isin(['06-01', '06-06', '08-15']).astype(int)
test['is_holiday'] = ((test['is_holiday'] == 1) | (test['dayofweek'] >= 5)).astype(int)
# DI(불쾌지수) 생성
#test['DI'] = 1.8 * test['기온(C)'] - 0.55 * (1 - test['습도(%)']/100) * (1.8 * test['기온(C)'] - 26) + 32

from sklearn.model_selection import TimeSeriesSplit

# 1. SMAPE 평가 지표 함수 정의
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred)))

# 2. TimeSeriesSplit을 위해 데이터 정렬 (핵심!)
# 시계열 순서가 섞이지 않도록 일시 및 건물번호 기준으로 정렬합니다.
train = train.sort_values(['일시', '건물번호']).reset_index(drop=True)

X_train = train[features]
y_train = train['전력소비량(kWh)']
X_test = test[features]

print(f"✅ 데이터 준비 완료! (Train: {X_train.shape}, Test: {X_test.shape})")

# 3. TimeSeriesSplit 설정
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

# 폴드별 예측값을 누적할 배열 (OOF 앙상블)
test_preds = np.zeros(len(X_test))
val_scores = []

print("🚀 TimeSeriesSplit 교차 검증 시작...")

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    # 훈련용/검증용 데이터 분할
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # 모델 정의
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        random_state=42,
        tree_method='hist',
        n_jobs=-1,
        enable_categorical=True
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False # 로그가 너무 길어지는 것 방지
    )
    
    # 검증 세트 예측 및 SMAPE 계산
    val_preds = model.predict(X_val)
    fold_smape = smape(y_val, val_preds)
    val_scores.append(fold_smape)
    
    print(f"Fold {fold+1} SMAPE: {fold_smape:.4f}")
    
    # 테스트 세트 예측 (각 폴드의 예측값을 평균내어 최종 예측 앙상블)
    test_preds += model.predict(X_test) / n_splits

print(f"\n✨ 교차 검증 완료! 평균 Val SMAPE: {np.mean(val_scores):.4f}")

# [6] 제출용 파일 생성
# [6] 제출용 파일 생성
submission = pd.read_csv('sample_submission.csv') 
submission['answer'] = test_preds

# 파일 저장 (이 코드를 복사해서 쓰세요!)
submission.to_csv('team_ex14.csv', index=False)

print("🏁 제출 파일(team_ex14.csv)이 생성되었습니다!") 

✅ 데이터 준비 완료! (Train: (203904, 24), Test: (16800, 24))
🚀 TimeSeriesSplit 교차 검증 시작...
Fold 1 SMAPE: 11.9099
Fold 2 SMAPE: 13.7295
Fold 3 SMAPE: 8.1471
Fold 4 SMAPE: 8.6318
Fold 5 SMAPE: 7.5493

✨ 교차 검증 완료! 평균 Val SMAPE: 9.9935
🏁 제출 파일(team_ex11.csv)이 생성되었습니다!
